# Notebook 1 – AXI-Lite Register Access Example

> ⚙️ **Prerequisite:** Complete [Notebook 0 – F2 Platform and Environment Setup](00_introduction.ipynb) before proceeding.

> ⚠️ **Cost reminder:** When working with this notebook in a workshop environment, remember to stop or terminate the F2 instance from the [EC2 Console](https://console.aws.amazon.com/ec2/) at the end of your session to avoid unnecessary costs.

This notebook covers loading a pre-built AFI onto an f2.6xlarge instance, using the AWS Cython bindings to read and write hardware registers over the OCL AXI-Lite interface, walking through the AXI-Lite protocol that carries those transactions, and validating that the FPGA computes correct results — all from Python running directly on the F2 instance.

### Learning Objectives

By the end of this notebook, you will be able to:

1. **Load an AFI and interact with FPGA registers from Python** using the Cython bindings to poke/peek CL registers over the OCL interface.
2. **Understand AXI-Lite**, the protocol that connects host software to FPGA hardware, including its five channels and valid/ready handshake.
3. **Implement the write, trigger, poll, read pattern**, a host-to-accelerator communication flow that scales to any register-mapped design.

## Table of Contents

- [1. CL_AXIL_REG_ACCESS Overview and Setup](#1.-CL_AXIL_REG_ACCESS-Overview-and-Setup)
- [2. Understanding AXI-Lite](#2.-Understanding-AXI-Lite:-The-Protocol-Behind-Register-Access)
- [3. Exploring the Register Map](#3.-Exploring-the-Register-Map)
- [4. Single Addition](#4.-Single-Addition)
- [5. Forced Carry Test](#5.-Forced-Carry-Test-(5-iterations))
- [6. Misaligned Access](#6.-Misaligned-Access)
- [7. Cleanup](#7.-Cleanup)

## 1. CL_AXIL_REG_ACCESS Overview and Setup

This section covers the **cl_axil_reg_access** custom logic design, which implements a simple 32-bit adder accessible via the **OCL AXI-Lite** interface (PCIe PF0, BAR0). The CL design is intentionally simple. The goal is to demonstrate the **host-to-FPGA communication** pattern: write configuration, trigger execution, poll for completion, read results.

The host drives the adder through a six-step register sequence:

1. Write `ADDR_OPERAND_A` with the first operand
2. Write `ADDR_OPERAND_B` with the second operand
3. Write `ADDR_CONTROL_STATUS` — set bit[0] (Start) to trigger the addition
4. Poll `ADDR_CONTROL_STATUS` — wait until bit[1] (Ready) is set by hardware
5. Read `ADDR_SUM`
6. Read `ADDR_CARRY` — reading both clears Ready

Unmapped addresses return `0xDEADBEEF`.

Source code: [cl_axil_reg_access on GitHub](https://github.com/aws/aws-fpga/tree/f2/hdk/cl/examples/cl_demo/cl_axil_reg_access/design/cl_axil_reg_access.sv)

> Execute the cell below to define the CL register map constants. These must be defined before any hardware interaction.

In [ ]:
# --- CL Register Map ---
ADDR_OPERAND_A      = 0x0000        # RW : 32-bit unsigned operand A
ADDR_OPERAND_B      = 0x0004        # RW : 32-bit unsigned operand B
ADDR_SUM            = 0x0008        # RO : 32-bit addition result
ADDR_CARRY          = 0x000C        # RO : Bit[0]: carry flag
ADDR_CONTROL_STATUS = 0x0010        # RW : Bit[0]: Start, Bit[1]: Ready (RO)

INVALID_ADDR_RESP   = 0xDEADBEEF    # Invalid address

MASK_CONTROL_READY = 0x02
MASK_CONTROL_START = 0x01

### Setup and Initialization

Execute the cells below to import the Cython bindings, load the cl_axil_reg_access AFI, and attach to the OCL BAR. This notebook uses an **f2.6xlarge** instance: 1 FPGA at **slot 0**, 24 vCPUs.

In [ ]:
import json
import random

from fpga_mgmt_wrapper import FpgaMgmt
from fpga_pci_wrapper import FpgaPCI
from helpers.notebook_utils import poll_register, print_fpga_info

print("✅ FPGA SDK bindings loaded")

In [ ]:
# Initialize the FPGA management and PCI access objects (connects to the SDK)
mgmt = FpgaMgmt()
pci = FpgaPCI()

SLOT_ID = 0
AGFI_ID = "agfi-06447dea0ca9b0a39"  # Replace with your cl_axil_reg_access AGFI

In [ ]:
# Load the cl_axil_reg_access AFI
# Note: load_local_image_sync_flags accepts an AGFI string via the afi_id parameter
load_result = mgmt.load_local_image_sync_flags(slot_id=SLOT_ID, afi_id=AGFI_ID)
print_fpga_info(load_result)

In [ ]:
# Attach to OCL BAR (PF0, BAR0)
handle = pci.pci_attach(slot_id=SLOT_ID, pf_id=0, bar_id=0, flags=0)
print(f"✅ Attached to OCL BAR, handle: {handle}")

> **Troubleshooting:** If any cell above fails, verify that the AFI loaded successfully (`status` should be `"loaded"`), and that `pci_attach` returned a valid handle (non-negative integer). If errors persist, restart the Jupyter kernel (**Kernel → Restart**) and re-run the setup cells sequentially from the top of this section.


## 2. Understanding AXI-Lite: The Protocol Behind Register Access

**AXI** (Advanced eXtensible Interface) is an ARM AMBA bus protocol used for communication between the Shell and the CL. **AXI-Lite** is the simplified variant designed for low-throughput register access. It is the protocol used by the OCL and SDA interfaces.

In this notebook, AXI-Lite is used through the OCL interface to read and write the registers defined above.

AXI-Lite defines **five independent channels**. The table below shows them as they appear on the OCL interface:

| Channel | Name | Direction | Purpose |
|---------|------|----------------------|--------|
| **AW** | Write Address | Requester (Shell) to Completer (CL) | Carries the target register address (AWADDR) |
| **W** | Write Data | Requester (Shell) to Completer (CL) | Carries the 32-bit data value (WDATA) and byte strobes (WSTRB) |
| **B** | Write Response | Completer (CL) to Requester (Shell) | Completer acknowledges the write with a status code (BRESP) |
| **AR** | Read Address | Requester (Shell) to Completer (CL) | Carries the target register address (ARADDR) |
| **R** | Read Data | Completer (CL) to Requester (Shell) | Completer returns the 32-bit data value (RDATA) and status (RRESP) |

Every channel uses a **valid/ready handshake**: the sender asserts its VALID signal (e.g., AWVALID), the receiver asserts its READY signal (e.g., AWREADY), and the data transfer occurs on the rising clock edge where both signals are high simultaneously. This handshake mechanism allows each side to apply backpressure, so neither side is forced to accept data before it is ready.

### 2.1. Write a Register (pci_poke)

Execute the cell below to write the value 10 to the Operand_A register. The step-by-step breakdown below traces what happens across the full stack.

In [ ]:
pci.pci_poke(handle, ADDR_OPERAND_A, 0x0000000A)
print(f"pci_poke → wrote 0x0000000A to ADDR_OPERAND_A")


1. **Python layer**: `pci.pci_poke(handle, 0x0000, 0x0000000A)` is called on the FpgaPCI object.

2. **Cython binding**: The Cython wrapper calls the C function `fpga_pci_poke()` in the AWS FPGA SDK library (`libfpga_pci.so`).

3. **C SDK library**: The library performs a **memory-mapped I/O (MMIO) write** to the virtual address corresponding to PCIe AppPF BAR0, offset `0x0000`.

4. **Linux kernel**: The kernel translates this MMIO write into a **PCIe Memory Write TLP** (Transaction Layer Packet), a formatted packet that carries the address and data across the PCIe link.

5. **PCIe physical link**: The TLP travels over the PCIe Gen4 x8 bus from the host CPU to the FPGA.

6. **Shell PCIe endpoint**: The Shell's PCIe core receives the TLP, recognizes it as targeting BAR0, and converts it into an **AXI-Lite write transaction** on the OCL interface.

7. **AXI-Lite Write Address channel (AW)**: The Shell drives AWVALID high with AWADDR = `0x0000`. Your CL asserts AWREADY when it can accept the address. The address transfers on the clock edge where both are high.

8. **AXI-Lite Write Data channel (W)**: The Shell drives WVALID high with WDATA = `0x0000000A` and `WSTRB = 4'b1111` (all four bytes valid). Your CL asserts WREADY. The data transfers on the handshake.

9. **Your CL logic**: The AXI-Lite slave module state machine in your RTL decodes the address, determines that it maps to the Operand_A register, and stores the value `0x0000000A` into that register on the next clock edge.

10. **AXI-Lite Write Response channel (B)**: Your CL drives BVALID high with BRESP = 2'b00 (OKAY). The Shell asserts BREADY. The response transfers on the handshake.

11. **Return path**: The Shell propagates the completion status back through PCIe to the host. The C library call returns, the Cython wrapper returns, and the Python call completes.

From the software perspective, this entire sequence appears as a single function call. From the hardware perspective, it is a single AXI-Lite write handshake lasting a few clock cycles at 250 MHz.

### 2.2. Read a Register (pci_peek)

Execute the cell below to read back the value written above. The step-by-step breakdown traces the read path.


In [ ]:
value = pci.pci_peek(handle, ADDR_OPERAND_A)
print(f"pci_peek → read 0x{value:08X} from ADDR_OPERAND_A")

1. **Python layer**: `pci.pci_peek(handle, 0x0000)` is called.

2. **Cython, C library, MMIO read**: The SDK performs a memory-mapped I/O read from BAR0 offset `0x0000`. Unlike a write, a **read is a blocking operation**: the CPU stalls until the data comes back from the FPGA.

3. **PCIe Memory Read TLP**: The kernel generates a PCIe read request TLP and sends it to the FPGA.

4. **Shell to AXI-Lite Read Address channel (AR)**: The Shell drives ARVALID high with ARADDR = `0x0000`. Your CL asserts ARREADY. The address transfers on the handshake.

5. **Your CL logic**: The AXI-Lite slave module decodes address `0x0000` as the Operand_A register and places the current value (`0x0000000A`) onto the Read Data channel.

6. **AXI-Lite Read Data channel (R)**: Your CL drives RVALID high with RDATA = `0x0000000A` and RRESP = 2'b00 (OKAY). The Shell asserts RREADY. The data transfers on the handshake.

7. **PCIe Completion TLP**: The Shell packages the read data into a PCIe Completion TLP and sends it back to the host over the PCIe link.

8. **Return path**: The CPU receives the completion, the MMIO read returns, the C library returns the 32-bit value, Cython wraps it as a Python integer, and the variable `value` holds `10`.

The key difference from a write: reads are **synchronous from the host's perspective**. The CPU issues the read request and waits (stalls) until the FPGA responds. This round-trip latency is why high-performance FPGA designs prefer high-throughput AXI4 burst transfers (via PCIS/PCIM) over individual register reads for large data movements.


## 3. Exploring the Register Map

Before running any additions, verify basic register access:
- Write to the RW registers and read them back.
- Read from an invalid address to observe the `0xDEADBEEF` sentinel.

In [ ]:
# Write and read back Operand_A
test_val_a = 0x0000_CAFE
pci.pci_poke(handle, ADDR_OPERAND_A, test_val_a)
readback_a = pci.pci_peek(handle, ADDR_OPERAND_A)
print(f"Operand_A: wrote 0x{test_val_a:08X}, read 0x{readback_a:08X}  {'✅ Match' if readback_a == test_val_a else '❌ Mismatch'}")

In [ ]:
# Write and read back Operand_B
test_val_b = 0xBEEF_0000
pci.pci_poke(handle, ADDR_OPERAND_B, test_val_b)
readback_b = pci.pci_peek(handle, ADDR_OPERAND_B)
print(f"Operand_B: wrote 0x{test_val_b:08X}, read 0x{readback_b:08X}  {'✅ Match' if readback_b == test_val_b else '❌ Mismatch'}")

In [ ]:
# Read from an invalid address (should return 0xDEADBEEF)
INVALID_ADDR = 0x0020
invalid_read = pci.pci_peek(handle, INVALID_ADDR)
print(f"Read from invalid address 0x{INVALID_ADDR:04X}: 0x{invalid_read:08X}  {'✅ Expected sentinel' if invalid_read == 0xDEADBEEF else '❌ Unexpected value'}")

The AXI-Lite interface is working:
- RW registers store and return the values written.
- Invalid addresses return the `0xDEADBEEF` sentinel, which is how the RTL signals an unmapped register.

## 4. Single Addition

The following cells execute a complete addition using the register sequence defined in Section 1. Modify the operand values or reorder the steps to observe how the CL design responds.

In [ ]:
# Define the operands
operand_a = 10
operand_b = 20

In [ ]:
# Step 1: Write operands
pci.pci_poke(handle, ADDR_OPERAND_A, operand_a)
pci.pci_poke(handle, ADDR_OPERAND_B, operand_b)
print(f"Wrote Operand_A = {operand_a}, Operand_B = {operand_b}")

In [ ]:
# Step 2: Trigger addition (write Start bit = 1)
pci.pci_poke(handle, ADDR_CONTROL_STATUS, MASK_CONTROL_START)
print("Triggered addition (Start = 1)")

In [ ]:
# Step 3: Poll Ready bit (bit[1] of Control_Status)
status = poll_register(pci, handle, ADDR_CONTROL_STATUS, MASK_CONTROL_READY)
if status != -1:
    print(f"Ready, Control_Status = 0x{status:08X}")

In [ ]:
# Step 4: Read results
hw_sum   = pci.pci_peek(handle, ADDR_SUM)
hw_carry = pci.pci_peek(handle, ADDR_CARRY)

In [ ]:
# Step 5: Validate against Python
expected = operand_a + operand_b
expected_sum   = expected & 0xFFFFFFFF
expected_carry = (expected >> 32) & 0x1
print(f"\nResult:   Sum = {hw_sum} (0x{hw_sum:08X}), Carry = {hw_carry}")
print(f"Expected: Sum = {expected_sum} (0x{expected_sum:08X}), Carry = {expected_carry}")
print(f"{'✅ PASS' if hw_sum == expected_sum and hw_carry == expected_carry else '❌ FAIL'}")

The complete register handshake:
1. Write operands to their registers.
2. Write `1` to the Start bit in Control_Status.
3. Poll until the Ready bit (bit[1]) goes high.
4. Read Sum and Carry. Reading both **clears** the Ready bit automatically.

## 5. Forced Carry Test (5 iterations)

Each iteration forces a carry by setting `a = rand() + 1` (minimum 1) and `b = 0xFFFFFFFF`. The carry bit must be set in every result.

In [ ]:
NUM_ITERS = 5
pass_count = 0

for i in range(NUM_ITERS):
    a = random.randint(1, 0xFFFFFFFF)
    b = 0xFFFFFFFF

    # Write operands
    pci.pci_poke(handle, ADDR_OPERAND_A, a)
    pci.pci_poke(handle, ADDR_OPERAND_B, b)

    # Trigger
    pci.pci_poke(handle, ADDR_CONTROL_STATUS, MASK_CONTROL_START)

    # Poll Ready
    poll_register(pci, handle, ADDR_CONTROL_STATUS, MASK_CONTROL_READY)

    # Read results
    hw_sum   = pci.pci_peek(handle, ADDR_SUM)
    hw_carry = pci.pci_peek(handle, ADDR_CARRY)

    # Validate: carry must always be 1
    expected = a + b
    exp_sum   = expected & 0xFFFFFFFF
    exp_carry = 1

    result_correct = (hw_sum == exp_sum) and (hw_carry == exp_carry)
    pass_count += result_correct

    status = "✅" if result_correct else "❌"
    print(f"[{i}] {status}  0x{a:08X} + 0x{b:08X} = 0x{hw_sum:08X}, carry={hw_carry}")

print(f"\n{'='*60}")
print(f"Results: {pass_count}/{NUM_ITERS} passed (all should have carry=1)")

## 6. Misaligned Access

AXI-Lite is a **32-bit aligned** interface. Reading from misaligned offsets produces behavior that depends on the RTL implementation. In this design, the hardware performs **byte rotation** based on the byte offset within the word. This section is observational with no pass/fail validation.

Write a known value to Operand_A and read from byte offsets 0, 1, 2, and 3 to observe the rotation.

In [ ]:
pattern = 0xAABBCCDD
pci.pci_poke(handle, ADDR_OPERAND_A, pattern)
print(f"Wrote 0x{pattern:08X} to Operand_A (0x{ADDR_OPERAND_A:04X})\n")

print(f"{'Offset':<10} {'Read Value':<14} {'Notes'}")
print("-" * 45)
for byte_offset in range(4):
    addr = ADDR_OPERAND_A + byte_offset
    val = pci.pci_peek(handle, addr)
    note = "aligned" if byte_offset == 0 else f"misaligned (+{byte_offset})"
    print(f"0x{addr:04X}     0x{val:08X}     {note}")

The bytes rotate as the offset shifts. The hardware still returns 32 bits, but the byte lane alignment changes. In practice, always use **aligned 4-byte offsets** (`0x0`, `0x4`, `0x8`, ...) for AXI-Lite register access. Misaligned reads are shown here for educational purposes; relying on them in production code is undefined behavior that varies across RTL implementations.

## 7. Cleanup

Detach the PCI handle and clear the AFI to release all resources.


In [ ]:
pci.pci_detach(handle)
print("✅ PCI handle detached")

mgmt.clear_local_image(SLOT_ID)
print("✅ AFI cleared from slot")

print("\nDone! You have completed Notebook 1.")

> ⚠️ **Cost reminder:** When working with this notebook in a workshop environment, remember to stop or terminate the F2 instance from the [EC2 Console](https://console.aws.amazon.com/ec2/) at the end of your session to avoid unnecessary costs.